### Importing Required Packages

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

### Reading the Dataset


In [0]:
batsmen_statistics_table = spark.table("workspace.default.Batsmen_Statistics_Transformed")

In [0]:
batsmen_statistics_table.limit(1).display()

### Cleaning invalid values for column values

#### Cleaned Average and Dot ball Columns

In [0]:
dot_ball_percentage_in_powerplay_column_cleaning = (
batsmen_statistics_table.withColumn("Dot_ball_percentage_in_powerplay",
                                    when(col("Dot_ball_percentage_in_powerplay") == 0, None)
                                    .otherwise(col("Dot_ball_percentage_in_powerplay"))
                                    )
)

In [0]:
dropping_zero_average_batsmen = (
    dot_ball_percentage_in_powerplay_column_cleaning.filter(col("Batsman_average") != 0)
)

In [0]:
dot_ball_percentage_in_middle_overs_cleaning = (
    dropping_zero_average_batsmen.withColumn("Dot_ball_percentage_in_middle_overs",
                                            when(col("Dot_ball_percentage_in_middle_overs") == 0, None)
                                            .otherwise(col("Dot_ball_percentage_in_middle_overs")))
                                     
)

In [0]:
dot_ball_percentage_in_death_overs_cleaning = (
    dot_ball_percentage_in_middle_overs_cleaning.withColumn("Dot_ball_percentage_in_death_overs",
                                                            when(col("Dot_ball_percentage_in_death_overs") == 0 , None).otherwise(col("Dot_ball_percentage_in_death_overs"))
                                                            )
)

In [0]:
dot_ball_percentage_in_death_overs_cleaning.filter(col("Dot_ball_percentage_in_death_overs").isNull()).display()

In [0]:
dot_ball_percentage_in_death_overs_cleaning.display()

#### Cleaned Average and Strike rate columns

In [0]:
batsman_average_in_difference_phases_cleaning = (
dot_ball_percentage_in_death_overs_cleaning.withColumn("batsman_average_in_powerplay",
                                        when(col("Batsman_average_in_powerplay") == 0, None)\
                                            .otherwise(col("Batsman_average_in_powerplay"))
                                        )\
.withColumn("batsman_average_in_middle_overs",
            when(col("Batsman_average_in_middle_overs") == 0, None)\
                .otherwise(col("Batsman_average_in_middle_overs"))
            )\
.withColumn("batsman_average_in_death_overs",
            when(col("Batsman_average_in_death_overs") == 0, None)\
                .otherwise(col("Batsman_average_in_death_overs"))

)\
)


                                        
                                                       

In [0]:
batsman_average_in_difference_phases_cleaning.display()

#### Filtering the DataFrame based on balls faced (min 100) or someone scored a half century

In [0]:
cleaned_batsmen_statistics_dataframe = (
batsman_average_in_difference_phases_cleaning.filter((col("total_balls_faced") >= 75) | (col("50+") >=1 )).sort("total_balls_faced")
)

In [0]:
cleaned_batsmen_statistics_dataframe.distinct().count()

In [0]:
cleaned_batsmen_statistics_dataframe.write\
    .format("delta")\
        .mode("overwrite")\
            .saveAsTable("Batsmen_Statistics_Table_Cleaned")

In [0]:
spark.read.table("Batsmen_Statistics_Table_Cleaned").display()